In [1]:
from functools import cache
from pathlib import Path

import pandas as pd
import google.auth

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis import utils
from calitp_data_analysis.sql import get_engine

from shared_utils import bq_utils
from update_vars import GTFS_DATA_DICT, analysis_month, file_name, last_year, previous_month

import _prep_gtfs_data
# Initialize credentials and DB engine
credentials, project = google.auth.default()
db_engine = get_engine()

In [2]:
@cache
def gcs_pandas():
    return GCSPandas()

In [3]:
PROD_PROJECT = "cal-itp-data-infra"
PROD_MART = "mart_gtfs_rollup"
MONTH_DATE_COL = "month_first_day"


In [5]:
df = bq_utils.download_table(
        project_name=PROD_PROJECT,
        dataset_name="tiffany_mart_gtfs_rollup",
        table_name=GTFS_DATA_DICT.gtfs_digest_rollup.schedule_rt_route_direction,
        date_col=MONTH_DATE_COL,
        start_date=last_year,
        end_date=analysis_month,
    )

GenericGBQException: Reason: 404 POST https://bigquery.googleapis.com/bigquery/v2/projects/cal-itp-data-infra/queries?prettyPrint=false: Not found: Dataset cal-itp-data-infra:tiffany_mart_gtfs_rollup was not found in location US

In [ ]:
df.columns

In [ ]:
crosswalk_url = f"{GTFS_DATA_DICT.gcs_paths.DIGEST_GCS}processed/{GTFS_DATA_DICT.gtfs_digest_rollup.crosswalk}_{file_name}.parquet"

In [ ]:
crosswalk_df = gcs_pandas().read_parquet(crosswalk_url)[["name", "analysis_name"]].drop_duplicates()

## `_prep_gtfs_data`

In [ ]:
schedule_rt_route_direction_summary_df = gcs_pandas().read_parquet(
        f"{GTFS_DATA_DICT.gcs_paths.DIGEST_GCS}raw/"
        f"{GTFS_DATA_DICT.gtfs_digest_rollup.schedule_rt_route_direction}_{file_name}.parquet"
    )

In [ ]:
GTFS_DATA_DICT.gcs_paths.DIGEST_GCS

In [ ]:
GTFS_DATA_DICT.gtfs_digest_rollup.schedule_rt_route_direction

In [ ]:
schedule_rt_route_direction_summary_df.columns

In [ ]:
GTFS_DATA_DICT.gtfs_digest_rollup.schedule_rt_route_direction

In [ ]:
def prep_operator_summary(file_name: str) -> pd.DataFrame:
    df = gcs_pandas().read_parquet(
        f"{GTFS_DATA_DICT.gcs_paths.DIGEST_GCS}raw/"
        f"{GTFS_DATA_DICT.gtfs_digest_rollup.operator_summary}_{file_name}.parquet"
    )

    # Select relevant columns
    df2 = df[
        [
            "month_first_day", "analysis_name", "caltrans_district", "vp_name", "tu_name",
            "n_trips", "day_type", "daily_trips", "ttl_service_hours", "n_routes", "n_days",
            "n_shapes", "n_stops", "vp_messages_per_minute", "n_vp_trips", "daily_vp_trips",
            "pct_vp_trips", "tu_messages_per_minute",
            "n_tu_trips", "daily_tu_trips", "pct_tu_trips", 
        ]
    ]
    
    # Multiply percetnage columns by 100. Clip any values above 100.
    df2.pct_tu_trips = df2.pct_tu_trips * 100
    df2.pct_vp_trips = df2.pct_vp_trips * 100
    df2.pct_tu_trips = df2.pct_tu_trips.clip(upper=100.0)
    df2.pct_vp_trips = df2.pct_vp_trips.clip(upper=100.0)
    
    # Clean columns
    df2.columns = df2.columns.str.replace("_", " ").str.title()
    df2 = df2.rename(columns={"Month First Day": "Date"})
    df2.columns = df2.columns.str.replace("Vp", "VP").str.replace("Tu", "TU")

    # Save processed file
    gcs_pandas().data_frame_to_parquet(
        df2,
        f"{GTFS_DATA_DICT.gcs_paths.DIGEST_GCS}processed/"
        f"{GTFS_DATA_DICT.gtfs_digest_rollup.operator_summary}_{file_name}.parquet"
    )

    return df2

In [ ]:
operator_summary = _prep_gtfs_data.prep_operator_summary(file_name)

In [ ]:
file_name

In [ ]:
monthly_route_summary = _prep_gtfs_data.prep_fct_monthly_routes(file_name)

In [ ]:
operator_hourly_summary = _prep_gtfs_data.prep_fct_operator_hourly_summary(file_name)